# KiCad + Freerouting 環境（routing reward 用）

**用 CPU runtime**（routing 是 CPU 工作，不需 GPU）。目的：
1. 裝 KiCad 9（`kicad-cli`）+ Java + Freerouting
2. **驗證我們生成的 .kicad_pcb 真的能被 KiCad 載入**（這是目前最關鍵的未知）

> KiCad 安裝 ~1.5GB、約 5 分鐘；session 斷了要重跑。Secrets 需 `GH_TOKEN`（clone repo）。

## 1. 裝 KiCad 9（含 kicad-cli）

In [ ]:
!apt-get -qq install -y software-properties-common > /dev/null
!add-apt-repository -y ppa:kicad/kicad-9.0-releases > /dev/null 2>&1
!apt-get -qq update > /dev/null
!apt-get -qq install -y --no-install-recommends kicad > /dev/null
!kicad-cli version

## 2. 裝 Java 25 + 抓最新 Freerouting
最新 Freerouting 用 Java 25 編譯（class 69），Colab 預設 Java 17 跑不動 → 裝 Temurin 25。
`--help` 是為了看清這版的 CLI 旗標（§7 route 要用）。

In [ ]:
# 裝 Temurin 25（Adoptium apt repo）
!apt-get -qq install -y wget gnupg lsb-release > /dev/null
!wget -qO - https://packages.adoptium.net/artifactory/api/gpg/key/public | gpg --dearmor | tee /etc/apt/trusted.gpg.d/adoptium.gpg > /dev/null
!echo "deb https://packages.adoptium.net/artifactory/deb $(lsb_release -cs) main" | tee /etc/apt/sources.list.d/adoptium.list > /dev/null
!apt-get -qq update > /dev/null
!apt-get -qq install -y temurin-25-jdk > /dev/null
!java -version

import json, urllib.request
rel = json.load(urllib.request.urlopen("https://api.github.com/repos/freerouting/freerouting/releases/latest"))
jar_url = next(a["browser_download_url"] for a in rel["assets"] if a["name"].endswith(".jar"))
print("freerouting", rel["tag_name"], "->", jar_url)
urllib.request.urlretrieve(jar_url, "/content/freerouting.jar")
!java -jar /content/freerouting.jar --help 2>&1 | head -30

## 3. 取得 repo

In [ ]:
import os, sys
from google.colab import userdata
GH = userdata.get("GH_TOKEN")
if not os.path.isdir("/content/genpcb"):
    !git clone https://{GH}@github.com/timmytsaa/genpcb.git /content/genpcb
else:
    !git -C /content/genpcb pull
!pip -q install -e /content/genpcb
sys.path.insert(0, "/content/genpcb/src")

## 4. 產一塊板 → .kicad_pcb

In [ ]:
from genpcb.data.procedural import generate_board
from genpcb.kicad import board_to_kicad_pcb

b = generate_board("mcu", seed=0)
open("/content/board.kicad_pcb", "w").write(board_to_kicad_pcb(b))
print("components:", len(b.components), "| nets:", len(b.nets))
!head -30 /content/board.kicad_pcb

## 5. 關鍵驗證：KiCad 載入得了嗎？
`kicad-cli pcb drc` 會試著載入並跑 DRC。**載入成功** = 我們的 writer 格式正確；
**失敗** = 格式要補東西，把錯誤訊息貼回來我調 writer。

In [ ]:
!kicad-cli pcb drc --format json --output /content/drc.json /content/board.kicad_pcb
import json
try:
    d = json.load(open("/content/drc.json"))
    v = d.get("violations", [])
    print(f"\n✅ KiCad 載入成功！DRC violations: {len(v)}")
    for x in v[:8]:
        print("  -", x.get("type"), "|", str(x.get("description"))[:80])
except Exception as e:
    print("\n❌ 載入/DRC 失敗：", e)
    print("→ 把上面 kicad-cli 的錯誤訊息整段貼給我，我調 board_to_kicad_pcb 的格式")

## 結論 / 下一步

- §5 KiCad 載入 ✅、§6 DSN 產出 ✅、§7 Freerouting 佈線、§8 routed_fraction、**§9 inline 佈線圖**。
- 封裝好的 ground truth：`from genpcb.kicad.route import routing_reward`（board → DSN → Freerouting → SES → routed_fraction）。**慢**（每板數十秒）→ 用於 audit / 訓 Model A surrogate，不直接進 GRPO inner loop。
- 更高畫質的佈線圖（之後）：SES → 併回 .kicad_pcb 的 track → `kicad-cli pcb export svg`。

把 §2 `--help`、§7、§8、§9 的結果貼給我（§9 就是你要的佈線圖）。

In [ ]:
from genpcb.kicad import board_to_dsn
open("/content/board.dsn", "w").write(board_to_dsn(b))   # b 來自 §4
print("DSN written")
!head -22 /content/board.dsn

## 7. Freerouting 佈線 → SES
Freerouting 需要顯示環境 → 用 `xvfb-run` 跑 headless。**CLI 旗標依版本不同**：
下面用 `-de`/`-do`（core 旗標）；若報錯，對照 §2 `--help` 調整。

In [ ]:
import os, subprocess
!apt-get -qq install -y xvfb > /dev/null

cmd = ["xvfb-run", "-a", "java", "-jar", "/content/freerouting.jar",
       "-de", "/content/board.dsn", "-do", "/content/board.ses", "-mp", "5"]
print("$", " ".join(cmd))
try:
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=900)
    print("STDOUT:\n", r.stdout[-1800:])
    print("STDERR:\n", r.stderr[-800:])
except subprocess.TimeoutExpired:
    print("timeout（900s）—— 板太大或 passes 太多")
print("SES 產生:", os.path.exists("/content/board.ses"))

In [ ]:
from genpcb.kicad import parse_ses_routed_fraction
dsn = open("/content/board.dsn").read()
ses = open("/content/board.ses").read() if os.path.exists("/content/board.ses") else ""
print(parse_ses_routed_fraction(dsn, ses))
if ses:
    print("\n--- SES 前 25 行 ---")
    print("\n".join(ses.splitlines()[:25]))

## 9. 看佈線圖（inline）
matplotlib 直接畫：板框 + 元件 pad（橘）+ 佈出的走線（F.Cu 紅 / B.Cu 藍）+ via（黑點）。
沒有 SES（佈線失敗）就只畫 placement。這是最快「看到 PCB 佈線」的方式。

In [ ]:
import matplotlib.pyplot as plt
from genpcb.kicad import plot_routing

ses = open("/content/board.ses").read() if os.path.exists("/content/board.ses") else None
plot_routing(b, ses_text=ses)   # b 來自 §4
plt.show()

## 結論 / 下一步

- 這本確認了 **(a) 環境裝好、(b) 我們的 .kicad_pcb 是否被 KiCad 接受**。
- 若 §5 載入成功 → 下一步我建 **IR → Specctra DSN 直接輸出**（不靠 pcbnew，避開 Colab 的 Python 綁定問題），接 Freerouting 佈線、解析 SES 算 `routed_fraction` → 這就是 routing reward。
- 若 §5 失敗 → 把錯誤貼來，先把 writer 修到能載入。

把 §2 的 `--help` 內容和 §5 的結果貼給我。